In [ ]:
# TODO why is x and y flipped

# PathoCellBench Cell Type Prediction and Evaluation

This notebook performs cell type prediction on the PathoCellBench dataset using CellWhisperer.
It uses the get_performance_metrics_left_vs_right function for evaluation.

In [ ]:
import torch
import numpy as np
import pandas as pd
import anndata
import json
import logging
from pathlib import Path
from typing import List

# Import CellWhisperer evaluation functions
from cellwhisperer.validation.zero_shot.functions import (
    get_performance_metrics_left_vs_right,
)
from cellwhisperer.utils.model_io import load_cellwhisperer_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Get paths from snakemake
model_path = snakemake.input.model
adata_path = snakemake.input.adata
image_path = snakemake.input.image

output_results = Path(snakemake.output.results)
output_per_class = Path(snakemake.output.per_class_metrics)
output_confusion = Path(snakemake.output.confusion_matrix)

seed = int(snakemake.wildcards.seed)
cell_type_level = 'coarse'

logger.info(f"Model: {model_path}")
logger.info(f"Data: {adata_path}")
logger.info(f"Seed: {seed}")

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [ ]:
# Load the model
logger.info("Loading CellWhisperer model...")
(
    pl_model,
    text_processor,
    transcriptome_processor,
    image_processor,
) = load_cellwhisperer_model(model_path=model_path, eval=True)
model = pl_model.model
logger.info("Model loaded successfully")

In [ ]:
# Load the PathoCellBench data
logger.info(f"Loading AnnData from {adata_path}")
adata = anndata.read_h5ad(adata_path)
logger.info(f"Loaded {adata.n_obs} cells with {adata.n_vars} genes")

# Select class set and labels from processed AnnData
counts_key = (
    "cell_type_counts" if cell_type_level == "fine" else "cell_type_counts_coarse"
)
label_col = "cell_type" if cell_type_level == "fine" else "cell_type_coarse"
counts_df = adata.obsm[counts_key]
class_names = list(counts_df.columns)
obs_labels = adata.obs[label_col].values
# No need to compute unique cell types; use columns of counts_df
logger.info(f"Using {len(class_names)} {cell_type_level} classes")

counts_df

In [ ]:
# Prepare classes and labels from processed AnnData (no aggregation here)
# Select classes and labels (use coarse mapping)
counts_key = "cell_type_counts_coarse"
label_col = "cell_type_coarse"
counts_df = adata.obsm[counts_key]
class_names = list(counts_df.columns)
obs_labels = adata.obs[label_col].values
ct_to_idx = {ct: i for i, ct in enumerate(class_names)}
correct_right_idx_per_left = [ct_to_idx[ct] for ct in obs_labels]
logger.info(f"Using {len(class_names)} {cell_type_level} classes")

In [ ]:
# Prepare text descriptions for each cell type
# These will be used as the "right" side in the evaluation
counts_key = "cell_type_counts_coarse"
label_col = "cell_type_coarse"
counts_df = adata.obsm[counts_key]
class_names = list(counts_df.columns)
obs_labels = adata.obs[label_col].values
ct_to_idx = {ct: i for i, ct in enumerate(class_names)}
correct_right_idx_per_left = [ct_to_idx[ct] for ct in obs_labels]
cell_type_descriptions = [
    f"A sample of {ct}" if ct.endswith("cells") else f"A sample of {ct} cells"
    for ct in class_names
]

logger.info(f"Created {len(cell_type_descriptions)} cell type descriptions")
for i, desc in enumerate(cell_type_descriptions[:3]):
    logger.info(f"  Example {i+1}: {desc}")

In [ ]:
cell_type_descriptions

In [ ]:
# Create mapping from cell type to index
cell_type_to_idx = ct_to_idx

# Create correct indices for evaluation
# For each cell (left), the correct cell type description (right) is at this index
correct_right_idx_per_left = [ct_to_idx[ct] for ct in obs_labels]

logger.info(f"Created {len(correct_right_idx_per_left)} correct index mappings")

In [ ]:
# Run evaluation using image data (set use_image_data=True)
logger.info("Running cell type prediction evaluation...")
logger.info(
    "This evaluates how well image embeddings match text descriptions of cell types"
)

metrics, per_class_df = get_performance_metrics_left_vs_right(
    model=model,
    left_input=adata,  # Image data will be extracted from adata
    right_input=cell_type_descriptions,  # Text descriptions of cell types
    correct_right_idx_per_left=correct_right_idx_per_left,
    average_mode=None,  # No averaging - evaluate at single-cell level
    grouping_keys=None,
    batch_size=128,
    score_norm_method="zscore",  # Normalize scores using z-score
    report_per_class_metrics=True,
    right_as_classes=True,
    use_image_data=True,  # Use image data instead of transcriptome
)

logger.info("Evaluation completed successfully")

In [ ]:
counts_df

In [ ]:
# Cross-entropy evaluation for patch mode
if snakemake.params.prediction_level == "patch":
    logger.info(
        "Computing cross-entropy between predicted and empirical distributions..."
    )

    # Get raw probability scores using existing infrastructure
    from cellwhisperer.utils.inference import score_left_vs_right

    # um... highly redundant it seems, as all of this has already been computed above....
    scores, _ = score_left_vs_right(
        left_input=adata,
        right_input=cell_type_descriptions,
        model=model,
        logit_scale=model.discriminator.temperature.exp(),
        average_mode=None,
        grouping_keys=None,
        batch_size=128,
        score_norm_method="softmax",  # Get probabilities
        use_image_data=True,
    )

    # Extract probabilities and empirical distributions
    predicted_probs = scores.T.cpu().numpy()  # Shape: (n_patches, n_cell_types)
    empirical_dists = counts_df.to_numpy()

    logger.info(f"Computing cross-entropy for {len(predicted_probs)} patches")

    # Cross-entropy calculation with numerical stability
    cross_entropies = []
    kl_divergences = []
    js_divergences = []

    for i in range(len(predicted_probs)):
        pred = predicted_probs[i] + 1e-8  # Numerical stability
        true = empirical_dists[i] + 1e-8

        # Normalize
        pred = pred / pred.sum()
        true = true / true.sum()

        # Cross-entropy: H(P, Q) = -sum(p_i * log(q_i))
        ce = -(true * np.log(pred)).sum()
        cross_entropies.append(ce)

        # KL divergence: D_KL(P||Q) = sum(p_i * log(p_i/q_i))
        kl = (true * np.log(true / pred)).sum()
        kl_divergences.append(kl)

        # Jensen-Shannon divergence (symmetric)
        from scipy.spatial.distance import jensenshannon

        js = jensenshannon(true, pred) ** 2
        js_divergences.append(js)

    # Store additional metrics for patch evaluation
    patch_metrics = {
        "mean_cross_entropy": float(np.mean(cross_entropies)),
        "std_cross_entropy": float(np.std(cross_entropies)),
        "mean_kl_divergence": float(np.mean(kl_divergences)),
        "mean_js_divergence": float(np.mean(js_divergences)),
        "entropy_predicted": float(
            np.mean([-(p * np.log(p + 1e-8)).sum() for p in predicted_probs])
        ),
        "entropy_empirical": float(
            np.mean([-(e * np.log(e + 1e-8)).sum() for e in empirical_dists])
        ),
        "probability_correlation": float(
            np.corrcoef(predicted_probs.flatten(), empirical_dists.flatten())[0, 1]
        ),
    }

    logger.info(f"Cross-entropy metrics:")
    logger.info(f"  Mean cross-entropy: {patch_metrics['mean_cross_entropy']:.4f}")
    logger.info(f"  Mean KL divergence: {patch_metrics['mean_kl_divergence']:.4f}")
    logger.info(f"  Mean JS divergence: {patch_metrics['mean_js_divergence']:.4f}")
    logger.info(
        f"  Probability correlation: {patch_metrics['probability_correlation']:.4f}"
    )

    # Store for later inclusion in results
    adata.uns["patch_metrics"] = patch_metrics
else:
    adata.uns["patch_metrics"] = {}

In [ ]:
# Extract and display key metrics
results = {
    "n_observations": int(adata.n_obs),  # cells or patches depending on mode
    "n_cell_types": len(counts_df.columns),
    "cell_types": list(counts_df.columns),
    "seed": seed,
    "prediction_level": snakemake.params.prediction_level,
}

# Add patch-specific metadata
if snakemake.params.prediction_level == "patch":
    results.update(
        {
            "patch_size": 224,
            "n_patches": int(adata.n_obs),
            "avg_cells_per_patch": float(adata.obs["n_cells"].mean()),
        }
    )
    # Add patch metrics
    results.update(adata.uns["patch_metrics"])
else:
    results["n_cells"] = int(adata.n_obs)

# Add macro-averaged metrics
for metric_name, metric_value in metrics.items():
    if isinstance(metric_value, torch.Tensor):
        results[metric_name] = float(metric_value.item())
    else:
        results[metric_name] = float(metric_value)

# Display key results
logger.info("\n=== Cell Type Classification Results ===")
logger.info(f"Accuracy (macro): {results.get('accuracy_macroAvg', 0):.4f}")
logger.info(f"F1 score (macro): {results.get('f1_macroAvg', 0):.4f}")
logger.info(f"Precision (macro): {results.get('precision_macroAvg', 0):.4f}")
logger.info(f"AUROC (macro): {results.get('rocauc_macroAvg', 0):.4f}")
logger.info(f"Recall@1 (macro): {results.get('recall_at_1_macroAvg', 0):.4f}")
logger.info(f"Recall@5 (macro): {results.get('recall_at_5_macroAvg', 0):.4f}")

# Display patch-specific metrics
if snakemake.params.prediction_level == "patch":
    logger.info(f"\n=== Patch-Level Cross-Entropy Metrics ===")
    logger.info(f"Number of patches: {results.get('n_patches', 0)}")
    logger.info(f"Average cells per patch: {results.get('avg_cells_per_patch', 0):.1f}")
    logger.info(f"Mean cross-entropy: {results.get('mean_cross_entropy', 0):.4f}")
    logger.info(f"Mean KL divergence: {results.get('mean_kl_divergence', 0):.4f}")
    logger.info(f"Mean JS divergence: {results.get('mean_js_divergence', 0):.4f}")
    logger.info(
        f"Probability correlation: {results.get('probability_correlation', 0):.4f}"
    )

In [ ]:
# Display per-class metrics
logger.info("\n=== Per-Class Metrics ===")
logger.info(f"\n{per_class_df.to_string()}")

# Identify best and worst performing cell types
best_f1 = per_class_df.nlargest(3, "f1")
worst_f1 = per_class_df.nsmallest(3, "f1")

logger.info("\n=== Best Performing Cell Types (by F1) ===")
logger.info(
    f"\n{best_f1[['f1', 'precision', 'accuracy', 'n_samples_in_class']].to_string()}"
)

logger.info("\n=== Worst Performing Cell Types (by F1) ===")
logger.info(
    f"\n{worst_f1[['f1', 'precision', 'accuracy', 'n_samples_in_class']].to_string()}"
)

In [ ]:
# Save results
output_results.parent.mkdir(parents=True, exist_ok=True)

with open(output_results, "w") as f:
    json.dump(results, f, indent=2)
logger.info(f"Saved results to {output_results}")

# Save per-class metrics
per_class_df.to_csv(output_per_class, index=True)
logger.info(f"Saved per-class metrics to {output_per_class}")

# Extract and save confusion matrix
confusion_cols = [
    col for col in per_class_df.columns if col.startswith("n_samples_predicted_as_")
]
if confusion_cols:
    confusion_df = per_class_df[confusion_cols]
    # Rename columns to remove prefix
    confusion_df.columns = [
        col.replace("n_samples_predicted_as_", "") for col in confusion_df.columns
    ]
    confusion_df.to_csv(output_confusion, index=True)
    logger.info(f"Saved confusion matrix to {output_confusion}")
else:
    logger.warning("No confusion matrix columns found in per_class_df")
    # Create an empty file
    pd.DataFrame().to_csv(output_confusion)

logger.info("Cell type prediction evaluation completed!")